In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from mapie.subsample import BlockBootstrap
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from var import DATA_OUT, START_DATE
from scintill_ai.conformal import enbpi_ts_regressor_predict, aci_ts_regressor_predict

## Data

In [ ]:
df = pd.read_pickle(Path(DATA_OUT, 'df.pickle'))

df['s4_mean_lag'] = df['s4_mean'].shift(5)
df['n_sat_lag'] = df['n_sat'].shift(5)

In [ ]:
# TRAIN_START, TRAIN_STOP = '2024-01-01', '2024-04-01'
# TEST_START, TEST_STOP = '2024-04-02', '2024-04-06'

TRAIN_START, TRAIN_STOP = '2022-09-06 07:00', '2022-10-10 15:59'
TEST_START, TEST_STOP = '2022-10-10 16:00', '2022-10-16 02:00'

In [ ]:
X_cols = [
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_mean_lag',
    'n_sat_lag',
]

y_col = 's4_mean'

X_train, X_test = df.loc[TRAIN_START:TRAIN_STOP, X_cols].copy(), df.loc[TEST_START:TEST_STOP, X_cols].copy()
y_train, y_test = df.loc[TRAIN_START:TRAIN_STOP, y_col].copy().fillna(0), df.loc[TEST_START:TEST_STOP, y_col].copy().fillna(0)

## Random Forest regressor

In [ ]:
# n_iter = 50
# n_splits = 5
# tscv = TimeSeriesSplit(n_splits=n_splits)
# random_state = 42
# rf_model = RandomForestRegressor(random_state=random_state)
# rf_params = {
#     "max_depth": [int(x) for x in np.linspace(2, 10, num=5)],
#     "n_estimators": [int(x) for x in np.linspace(10, 100, num=10)],
# }
# cv_obj = RandomizedSearchCV(
#     rf_model,
#     param_distributions=rf_params,
#     n_iter=n_iter,
#     cv=tscv,
#     scoring="neg_root_mean_squared_error",
#     random_state=random_state,
#     verbose=0,
#     n_jobs=-1,
# )
# cv_obj.fit(X_train, y_train.values)
# cv_obj.best_estimator_

In [ ]:
rf = RandomForestRegressor(
    # max_depth=6, n_estimators=60, random_state=42,
    max_depth=4, n_estimators=40, random_state=42,
)

## Conformal Prediction

In [ ]:
cv = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

### EnbPI (*without* update of residuals)

In [ ]:
enbpi_res = enbpi_ts_regressor_predict(
    model=rf, cv=cv, train_data=(X_train, y_train), test_data=(X_test, y_test)
)

### ACI (*without* update of residuals)

In [ ]:
aci_res = aci_ts_regressor_predict(
    model=rf,
    cv=cv,
    gamma=0.05,
    train_data=(X_train, y_train),
    test_data=(X_test, y_test),
)

### EnbPI (*with* update of residuals)

### ACI (*with* update of residuals)

## Plot

In [ ]:
plot_dict = aci_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, ls='-', label="Actual (test)", c="tab:orange")
ax.plot(
    y_test.index,
    plot_dict[0]["y_pred"],
    lw=1,
    ls=':',
    c="tab:blue",
    label="Forecast",
)

for i, result_ in enumerate(plot_dict):
    y_pis = result_["y_pis"]
    color = plt.cm.Blues(1 - i/len(plot_dict))
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.3,
        color=color,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('Random Forest + ACI', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y %H:%M'))
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)
ax.yaxis.grid(True, color='k', linewidth=0.2, alpha=0.4)
[ax.spines[s].set_visible(False) for s in ax.spines]
ax.legend(frameon=True, facecolor='white', edgecolor='none')
# ax.set_xlim(y_test.index[0], y_test.index[-1])
ax.set_xlim(y_test.loc['2022-10-10 18'].index[0], y_test.loc['2022-10-12 04'].index[0])
ax.set_ylim(0, 0.9)

# plt.savefig('aci.png', dpi=500, bbox_inches='tight')
plt.show()

In [ ]:
plot_dict = enbpi_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, ls='-', label="Actual (test)", c="tab:orange")
ax.plot(
    y_test.index,
    plot_dict[0]["y_pred"],
    lw=1,
    ls=':',
    c="tab:blue",
    label="Forecast",
)

for i, result_ in enumerate(plot_dict):
    y_pis = result_["y_pis"]
    color = plt.cm.Blues(1 - i/len(plot_dict))
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.3,
        color=color,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('Random Forest + EnbPI', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y %H:%M'))
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)
ax.yaxis.grid(True, color='k', linewidth=0.2, alpha=0.4)
[ax.spines[s].set_visible(False) for s in ax.spines]
ax.legend(frameon=True, facecolor='white', edgecolor='none')
ax.set_xlim(y_test.index[0], y_test.index[-1])
# ax.set_ylim(0, 0.5)

# plt.savefig('enbpi.png', dpi=500, bbox_inches='tight')
plt.show()

- implementa il metodo di [partial_fit](https://mapie.readthedocs.io/en/latest/examples_regression/4-tutorials/plot_ts-tutorial.html#estimate-prediction-intervals-on-the-test-set)
- scorri varie date per train/test (vedi l'altro notebook)
- crea una funzione per il plotting